In [40]:
import pandas as pd
import numpy as np
import time
import json
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [41]:
from ufal.udpipe import Model, Pipeline
import ufal.udpipe

In [42]:
# Configuration
MODEL_NAME = "UDPipe"
SAMPLE_TYPES = ["glosses", "medieval_charters"]
TASKS = ["lemmatization", "pos_tagging"]

# Notebook path
notebook_path = os.path.abspath("04-udpipe.ipynb")

In [43]:
# Results storage
results = {
    "model_name": MODEL_NAME,
    "processing_times": {},
    "accuracy": {},
    "precision": {},
    "recall": {},
    "f1_score": {}
}

In [44]:
def word_joiner(gold_df):
    # Extract text to reconstruct from words

    sample_texts = []
    for sample_id in gold_df['sample_id'].unique():
        words = gold_df[gold_df['sample_id'] == sample_id]['word'].tolist()
        text = ' '.join(words)
        sample_texts.append((sample_id, text))

    return sample_texts

In [47]:
def udpipe_processor(sample_texts, sample_type, pipeline):
    processed_results = []
    for sample_id, text in sample_texts:

        processed = pipeline.process(text)
        sentences = processed.strip().split('\n\n')

        for sent in sentences:
            lines = sent.split("\n")
            for idx, line in enumerate(lines):
                if line.startswith('#') or not line.strip():
                    continue

                fields = line.split('\t')
                if len(fields) != 10:
                    continue

                form = fields[1]
                lemma = fields[2]
                upos = fields[3]

                if sample_type == "glosses":
                    word_id = f"http://gams.uni-graz.at/o:glossvibe.bvi#{sample_id}.{idx}"
                else:
                    word_id = str(idx)

                processed_results.append({
                    "sample_id": sample_id,
                    "word_id": word_id,
                    "word": form,
                    "lemma": lemma,
                    "pos": upos
                })

    return processed_results


In [33]:
def analyse(merged_df, sample_type):
    # Convert the lemma columns to string type to ensure consistent comparison
    merged_df['lemma_gold'] = merged_df['lemma_gold'].astype(str)
    merged_df['lemma_pred'] = merged_df['lemma_pred'].astype(str)

    # Evaluate lemmatization
    lemma_accuracy = accuracy_score(merged_df['lemma_gold'], merged_df['lemma_pred'])

    # Create a binary array where True means the prediction matches the gold standard
    matches = merged_df['lemma_gold'] == merged_df['lemma_pred']

    # Fix the precision_recall_fscore_support call
    lemma_precision, lemma_recall, lemma_f1, _ = precision_recall_fscore_support(
        matches,
        [True] * len(merged_df),
        average='binary'
    )

    # Convert the lemma columns to string type to ensure consistent comparison
    merged_df['pos_gold'] = merged_df['pos_gold'].astype(str)
    merged_df['pos_pred'] = merged_df['pos_pred'].astype(str)

    # Evaluate lemmatization
    pos_accuracy = accuracy_score(merged_df['pos_gold'], merged_df['pos_pred'])

    # Create a binary array where True means the prediction matches the gold standard
    matches = merged_df['pos_gold'] == merged_df['pos_pred']

    # Fix the precision_recall_fscore_support call
    pos_precision, pos_recall, pos_f1, _ = precision_recall_fscore_support(
        matches,
        [True] * len(merged_df),
        average='binary'
    )

    results["accuracy"][f"{sample_type}_lemma"] = lemma_accuracy
    results["precision"][f"{sample_type}_lemma"] = lemma_precision
    results["recall"][f"{sample_type}_lemma"] = lemma_recall
    results["f1_score"][f"{sample_type}_lemma"] = lemma_f1

    results["accuracy"][f"{sample_type}_pos"] = pos_accuracy
    results["precision"][f"{sample_type}_pos"] = pos_precision
    results["recall"][f"{sample_type}_pos"] = pos_recall
    results["f1_score"][f"{sample_type}_pos"] = pos_f1

    merged_df.to_csv(f"../results/{MODEL_NAME}_{sample_type}_detailed.csv", index=False)

    print(f"Completed {sample_type}. Processing time: {processing_time:.2f}s")
    print(f"Lemmatization accuracy: {lemma_accuracy:.4f}")
    print(f"POS tagging accuracy: {pos_accuracy:.4f}")
    print("-" * 50)


In [34]:
from ufal.udpipe import Model, Pipeline
import ufal.udpipe

In [35]:
# Load the UDPipe model
model_path = "/Users/Thea/Desktop/LatinNLPTools/scripts/latin-ittb-ud-2.5-191206.udpipe"
model = Model.load(model_path)
if not model:
    raise Exception("Model not loaded!")

In [37]:
# Create a processing pipeline
pipeline = Pipeline(model, "tokenize", Pipeline.DEFAULT, Pipeline.DEFAULT, "conllu")

In [ ]:
for sample_type in SAMPLE_TYPES:
    print(f"Processing {sample_type}...")

    # Load gold standard data
    gold_file = os.path.join(os.path.dirname(notebook_path), f"../data/gold_standard/gs_{sample_type}.csv")

    gold_df = pd.read_csv(gold_file)

    # Extract text to reconstruct from words
    sample_texts = word_joiner(gold_df)

    # Process samples and measure time
    start_time = time.time()

    processed_results = udpipe_processor(sample_texts, sample_type, pipeline)

    processing_time = time.time() - start_time
    results["processing_times"][sample_type] = processing_time

    print(f"Data processes with {MODEL_NAME} in {processing_time} seconds.")

    # Merge gold_df with processed_results
    pred_df = pd.DataFrame(processed_results)

    print("Original gold_df length:", len(gold_df))
    print("Processed results length:", len(processed_results))
    print("Pred DF length:", len(pred_df))

    # Try a loose merge to inspect mismatches
    merged_preview = pd.merge(gold_df, pred_df, on='sample_id', how='left', suffixes=('_gold', '_pred'))
    print("Loose merge length (on sample_id only):", len(merged_preview))

    # Check some non-matching examples
    unmatched = gold_df[~gold_df['word'].isin(pred_df['word'])]
    print("Examples of unmatched words:", unmatched.head())


    merged_df = pd.merge(gold_df, pred_df, on=['sample_id', 'word_id'], suffixes=('_gold', '_pred'))


    print(merged_df)
    #print(f"Running analysis on {sample_type}...")
    #analyse(merged_df, sample_type)




Processing glosses...
Data processes with UDPipe in 0.1674048900604248 seconds.
Original gold_df length: 666
Processed results length: 668
Pred DF length: 668
Loose merge length (on sample_id only): 4197
Examples of unmatched words:       sample_id                                            word_id word  \
575   BVi.04a10  http://gams.uni-graz.at/o:glossvibe.bvi#BVi.04...  ·l·   
596  Ang.58a13b  http://gams.uni-graz.at/o:glossvibe.bvi#Ang.58...   s.   

                lemma  pos  
575  (Roman numerals)  NUM  
596             idest  NaN  
Processing medieval_charters...
Data processes with UDPipe in 8.12232780456543 seconds.
Original gold_df length: 24192
Processed results length: 24256
Pred DF length: 24256
Loose merge length (on sample_id only): 1392472
Examples of unmatched words:      sample_id word_id     word lemma    pos
360     dev-s9       1  [Propn]     _  PROPN
2096   dev-s77       3  [Propn]     _  PROPN
2110   dev-s78       3  [Propn]     _  PROPN
2124   dev-s79       3  

In [39]:
# Save summary results
with open(f"../results/{MODEL_NAME}_summary.json", "w") as f:
    json.dump(results, f, indent=2)